# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzayan-bit/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

In [3]:
con.sql("""
SHOW TABLES;
""").df()

,name


In [4]:
import duckdb

# Hardcoded token as requested
import huggingface_hub
HF_TOKEN = huggingface_hub.get_token()

con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

print('DuckDB connected and HF secret created!')


DuckDB connected and HF secret created!


In [5]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:,} rows")

dim_clients            104 rows
dim_content            519,606 rows
fact_daily             78,835,655 rows
fact_daily_sample      11,694,072 rows
fact_query_90d         2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis and Time Window
- **Unit of Analysis**: One row represents a single content page ().
- **Time Window**: We aggregate performance over a 90-day window (comparing the most recent 30 days vs the prior 30 days) to detect traffic decline.


unit_query = con.sql(f"""
    SELECT content_hash_id, COUNT(*) as days_active,
           MIN(report_date) as first_date, MAX(report_date) as last_date
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= '2026-01-01'
    GROUP BY content_hash_id
    LIMIT 5
""").df()
display(unit_query)


The target is future organic impressions during the prediction window. The model predicts future visibility rather than current performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Queries
- **Grain**: We expect  to be unique in our aggregated feature table and .
- **Missing Values**: We check how many content items are missing  to ensure it is usable.


In [6]:
missing_values = con.sql(f"""
    SELECT 
        COUNT(*) as total_content,
        COUNT(word_count) as non_null_word_count,
        COUNT(*) - COUNT(word_count) as missing_word_count
    FROM {TABLES["dim_content"]}
""").df()
display(missing_values)

grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(*) as row_count
    FROM {TABLES["dim_content"]}
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
""").df()
print(f"Content IDs with multiple rows in dim_content: {len(grain_check)}")


,total_content,non_null_word_count,missing_word_count
0,519606,341838,177768


Content IDs with multiple rows in dim_content: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits
- **Unbalanced History**: Clients started tracking at different times ().
- **GA4 Availability**: Many clients do not have GA4 access (), meaning engagement metrics are not universally available.


In [7]:
limits_query = con.sql(f"""
    SELECT has_ga4_access, COUNT(*) as client_count
    FROM {TABLES["dim_clients"]}
    GROUP BY has_ga4_access
""").df()
display(limits_query)

history_query = con.sql(f"""
    SELECT YEAR(gsc_data_start) as start_year, COUNT(*) as client_count
    FROM {TABLES["dim_clients"]}
    GROUP BY start_year
    ORDER BY start_year
""").df()
display(history_query)


,has_ga4_access,client_count
0,False,40
1,<NA>,10
2,True,54


,start_year,client_count
0,2025,40
1,2026,27
2,<NA>,37


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.